<div style="position:relative;overflow:hidden;border-radius:22px;min-height:280px;background:#0B1F3A url('https://images.unsplash.com/photo-1593941707882-a5bba14938c7?auto=format&fit=crop&w=1600&q=80') center 40%/cover;">
<div style="position:absolute;inset:0;background:linear-gradient(105deg,rgba(11,31,58,.92) 0%,rgba(11,31,58,.55) 48%,rgba(232,93,76,.28) 100%);"></div>
<div style="position:relative;color:#F6F1E7;padding:36px 40px;">
<p style="letter-spacing:.2em;text-transform:uppercase;font-size:12px;color:#F4C95D;margin:0;">Playground Series · S6E9 · train the right table</p>
<h1 style="margin:12px 0 10px;font-size:38px;line-height:1.08;text-shadow:0 2px 18px rgba(0,0,0,.45);">Same columns. Different generator.</h1>
<p style="margin:0;font-size:18px;max-width:42rem;line-height:1.45;color:#F6F1E7;">The original 10k EV table looks like this competition. Extra-train lost <b>all five folds</b> (public <b>0.94621</b>). This kernel trains the Playground switchboard and can write the verified <b>0.94650</b> ordering.</p>
<p style="margin:16px 0 0;font-size:13px;color:#00B4A6;">Den Pugovkin · leak-free LightGBM · original 10k is the diagnostic, not extra train</p>
</div>
</div>


<div style="background:#F6F1E7;border:1px solid #D6CDB8;border-radius:16px;padding:20px 24px;margin:16px 0;">
<p style="letter-spacing:.14em;text-transform:uppercase;font-size:11px;color:#E85D4C;margin:0 0 10px;">How to read this notebook</p>
<p style="margin:0 0 12px;line-height:1.55;color:#1C1917;">This is <a href="https://www.kaggle.com/competitions/playground-series-s6e9" style="color:#0B1F3A;"><b>Playground Series S6E9</b></a>: predict <code>P(Will_Buy_EV = Yes)</code>. Metric is <b>ROC AUC</b>. The analysis and trained file are ours; any community ordering used for the public file is named and checked explicitly.</p>
<p style="margin:0 0 12px;line-height:1.55;color:#1C1917;"><b>The hook.</b> Allowed original data with the same column names is still a <i>different purchase machine</i>. Lab <code>exp025</code> concatenated those rows onto fold-train only and lost <b>0/5 folds</b>.</p>
<p style="margin:0 0 12px;line-height:1.55;color:#1C1917;"><b>What this kernel does.</b> It plots that mismatch live, then trains <b>one leak-free LightGBM</b> on Playground rows only (digits + fold-local target rates). That family scored <b>CV 0.946 / public 0.94616–0.94628</b> in the lab. Original rows never enter a fit.</p>
<p style="margin:0;line-height:1.55;color:#1C1917;"><b>Files.</b> Always writes <code>submission_trained.csv</code>. Official <code>submission.csv</code> prefers Taeyang's structurally verified lexsort output (reported <b>0.94650</b>), then our pinned 0.94649 high-band stack, then the trained model. This is attribution, not a claim that 0.94650 is our model score.</p>
</div>


<div style="background:#F6F1E7;border-left:6px solid #E85D4C;padding:16px 20px;border-radius:0 14px 14px 0;margin:14px 0;">
<b>TL;DR</b>
<ol style="margin:8px 0 0;padding-left:1.15rem;line-height:1.55;">
<li>Playground <code>Subsidy=No</code> → <b>0.58%</b> Yes. The original 10k uses the same label and is a milder machine.</li>
<li>Extra-train on that table: frozen v1, 3-seed LightGBM, original rows never in validation. <b>0/5</b>. Public <b>0.94621</b>.</li>
<li>This kernel trains Playground-only LightGBM (same family as the honest <b>0.94628</b> bag).</li>
<li>For the <b>0.94650 candidate</b>, attach notebook output <code>taeyangg4/s6e9-094649-multi-paradigm-lexsort-master</code>. Its path, IDs, zero ties, rank grid, and known first rows are checked.</li>
<li>Fallback: attach nina <code>0.94649.csv</code> / <code>0.94644.csv</code> / <code>0.94639.csv</code>. Do not mix original 10k into either ordering.</li>
</ol>
</div>

CPU. Internet on for <code>great_tables</code>. Attach Playground S6E9 + <code>itzzomkar/ev-adoption-behavior-and-range-anxiety</code> + notebook output <code>taeyangg4/s6e9-094649-multi-paradigm-lexsort-master</code>. Optional fallback: <code>nina2025/ps-s6e9-11</code> and <code>nina2025/ps-s6e9-13</code>. About 15–25 minutes after the charts.


## Contents

1. [Two tables, one schema](#1) — names are not a process
2. [The switchboard does not transfer](#2) — live heatmaps
3. [Numerics disagree too](#3) — income and commute
4. [Why extra labelled rows can hurt](#4) — 0/5 folds, public 0.94621
5. [Train the Playground machine](#5) — leak-free LightGBM
6. [Write the public file](#6) — trained model, 0.94649 fallback, 0.94650 candidate


<h2 id="1" style="color:#0B1F3A;border-bottom:3px solid #E85D4C;padding-bottom:6px;">1. Two tables, one schema</h2>

Question: is the original 10k the same data-generating process, or just the same column names?


In [ ]:
%pip install -q great_tables


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from great_tables import GT, loc, style
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

INK, PAPER, NAVY = "#1C1917", "#F6F1E7", "#0B1F3A"
TEAL, GOLD, CORAL = "#00B4A6", "#F4C95D", "#E85D4C"
HEAT = [CORAL, PAPER, TEAL]
LAYOUT = dict(
    paper_bgcolor=PAPER, plot_bgcolor=PAPER,
    font=dict(color=INK, size=13),
    margin=dict(l=48, r=24, t=56, b=40), height=460,
)


def draw(fig):
    fig.show()


In [ ]:
def journal(frame, title, subtitle, note="", rowname=None, groupname=None):
    tbl = GT(frame, rowname_col=rowname, groupname_col=groupname)
    tbl = (
        tbl.tab_header(title=title, subtitle=subtitle)
        .tab_style(style.fill(color=NAVY), loc.header())
        .tab_style(style.text(color=PAPER), loc.header())
        .tab_style(style.fill(color="#123A4C"), loc.column_labels())
        .tab_style(style.text(color=PAPER), loc.column_labels())
        .tab_style(style.fill(color=TEAL), loc.row_groups())
        .tab_style(style.text(color=PAPER, weight="bold"), loc.row_groups())
        .tab_options(
            table_background_color=PAPER, table_font_color=INK,
            table_font_names=["Georgia", "Times New Roman", "serif"],
            table_border_top_color=CORAL, table_border_top_width="4px",
            heading_background_color=NAVY, heading_align="left",
            column_labels_background_color="#123A4C",
            row_group_background_color=TEAL,
            source_notes_background_color=PAPER,
            row_striping_include_table_body=True,
            row_striping_background_color="#EFE8D8",
            table_width="100%", data_row_padding="7px",
        )
    )
    return tbl.tab_source_note(note) if note else tbl


def show(tbl):
    display(HTML(tbl.as_raw_html()))


In [ ]:
def find_csv(*names: str) -> Path:
    root = Path("/kaggle/input")
    local = [
        Path("competitions/predicting-electric-vehicle-purchases/data/raw"),
        Path("data/raw"),
    ]
    roots = [root, *local] if root.exists() else local
    hits = []
    for base in roots:
        if not base.exists():
            continue
        for name in names:
            hits.extend(p for p in base.rglob(name) if p.is_file())
    if not hits:
        raise FileNotFoundError(" | ".join(names))
    return hits[0]


train = pd.read_csv(find_csv("train.csv"))
test = pd.read_csv(find_csv("test.csv"))
sample = pd.read_csv(find_csv("sample_submission.csv"))
orig_path = find_csv(
    "EV_Adoption_and_Range_Anxiety_Dataset.csv",
    "ev_adoption_and_range_anxiety_dataset.csv",
)
orig = pd.read_csv(orig_path)
y = (train["Will_Buy_EV"].astype(str) == "Yes").astype(int)
print("train", train.shape, "test", test.shape, "original", orig.shape)
print(orig_path)


In [ ]:
ALIASES = {
    "annual_income_usd": "Annual_Income_USD",
    "government_subsidy_availability": "Subsidy_Available",
    "government_subsidy": "Subsidy_Available",
    "range_anxiety_level": "Range_Anxiety_Level",
    "will_buy_ev": "Will_Buy_EV",
    "environmental_concern_level": "Environmental_Concern_Level",
}


def norm(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    rename = {}
    for col in out.columns:
        key = " ".join(str(col).lower().replace("-", "_").split()).replace(" ", "_")
        if key in ALIASES:
            rename[col] = ALIASES[key]
    return out.rename(columns=rename)


orig = norm(orig)
need = ["Subsidy_Available", "Range_Anxiety_Level", "Will_Buy_EV"]
missing_need = [c for c in need if c not in orig.columns]
if missing_need:
    raise KeyError(missing_need)
complete = orig.dropna(subset=need).copy()
print("original complete on switches", len(complete), "dropped", len(orig) - len(complete))


In [ ]:
def yes_rate(frame):
    return float((frame["Will_Buy_EV"].astype(str) == "Yes").mean())


def rate_at(frame, col, level):
    yes = (frame["Will_Buy_EV"].astype(str) == "Yes").astype(int)
    hit = frame.assign(yes=yes)
    hit = hit.loc[hit[col].astype(str) == level, "yes"]
    return float(hit.mean()) if len(hit) else float("nan")


snap = pd.DataFrame(
    {
        "table": ["Playground train", "Playground test", "Original 10k"],
        "rows": [len(train), len(test), len(orig)],
        "yes_rate": [yes_rate(train), np.nan, yes_rate(complete)],
        "missing_cells": [
            int(train.isna().sum().sum()),
            int(test.isna().sum().sum()),
            int(orig.isna().sum().sum()),
        ],
        "subsidy_no": [
            rate_at(train, "Subsidy_Available", "No"),
            np.nan,
            rate_at(complete, "Subsidy_Available", "No"),
        ],
    }
)
show(
    journal(
        snap, "Same labels. Different missingness. Different base rate.",
        "ROC AUC on P(Will_Buy_EV = Yes). Higher is better.",
        "Lab exp025 dropped 534 incomplete original rows before extra-train. Names still matched.",
        rowname="table",
    )
    .fmt_integer(["rows", "missing_cells"])
    .fmt_percent(["yes_rate", "subsidy_no"], decimals=2)
    .sub_missing(missing_text="—")
)


In [ ]:
def card(kicker, value, note, color):
    return (
        f'<div style="flex:1;min-width:150px;background:{PAPER};border-top:4px solid {color};'
        f'padding:14px 16px;border-radius:12px;box-shadow:0 8px 24px rgba(11,31,58,.08);">'
        f'<p style="margin:0;letter-spacing:.12em;text-transform:uppercase;font-size:11px;color:{color};">{kicker}</p>'
        f'<p style="margin:8px 0 0;font-size:26px;font-weight:700;color:{NAVY};">{value}</p>'
        f'<p style="margin:4px 0 0;font-size:12px;color:#78716C;">{note}</p></div>'
    )


display(HTML(
    '<div style="display:flex;gap:12px;flex-wrap:wrap;margin:8px 0 16px;">'
    + card("Playground Yes", f"{yes_rate(train):.2%}", f"{len(train):,} rows · clean", TEAL)
    + card("Original Yes", f"{yes_rate(complete):.2%}", f"{len(complete):,} complete rows", CORAL)
    + card("Subsidy=No · PG", f"{rate_at(train, 'Subsidy_Available', 'No'):.2%}", "kill switch", GOLD)
    + card("Subsidy=No · orig", f"{rate_at(complete, 'Subsidy_Available', 'No'):.2%}", "milder machine", CORAL)
    + "</div>"
))


Decision: Playground is a clean 668k probability table. Original is a smaller, messier table with the same names. Extra labelled rows from that file are not more of the same synthetic switch.


<h2 id="2" style="color:#0B1F3A;border-bottom:3px solid #E85D4C;padding-bottom:6px;">2. The switchboard does not transfer</h2>

Question: do subsidy and range anxiety mean the same thing on both tables?


In [ ]:
def board(frame: pd.DataFrame) -> pd.DataFrame:
    yes = (frame["Will_Buy_EV"].astype(str) == "Yes").astype(int)
    tab = (
        frame.assign(yes=yes)
        .groupby(["Range_Anxiety_Level", "Subsidy_Available"], observed=False)
        .yes.mean()
        .unstack("Subsidy_Available")
    )
    return tab.reindex(index=["High", "Medium", "Low"], columns=["No", "Yes"])


pg, og = board(train), board(complete)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Playground train · 668,665 rows", "Original 10k · complete rows"),
)
for col, grid, cbar in ((1, pg, False), (2, og, True)):
    fig.add_trace(
        go.Heatmap(
            z=grid.values, x=list(grid.columns.astype(str)), y=list(grid.index.astype(str)),
            colorscale=[[0, CORAL], [0.5, PAPER], [1, TEAL]], zmin=0, zmax=0.35,
            text=np.char.mod("%.2f%%", 100 * grid.values), texttemplate="%{text}",
            showscale=cbar, colorbar=dict(title="P(Yes)", tickformat=".0%"),
        ),
        row=1, col=col,
    )
fig.update_layout(**LAYOUT, title="Same 2 × 3 keys. Two purchase machines.")
fig.update_yaxes(autorange="reversed")
draw(fig)


In [ ]:
delta = pg - og
fig = go.Figure(
    go.Heatmap(
        z=delta.values, x=list(delta.columns.astype(str)), y=list(delta.index.astype(str)),
        colorscale=[[0, CORAL], [0.5, PAPER], [1, TEAL]], zmid=0,
        text=np.char.mod("%+.2f pp", 100 * delta.values), texttemplate="%{text}",
        colorbar=dict(title="PG − orig"),
    )
)
fig.update_layout(
    **LAYOUT, title="Where Playground is a kill switch and original is not",
)
fig.update_yaxes(autorange="reversed")
draw(fig)


In [ ]:
def rates(frame, col):
    yes = (frame["Will_Buy_EV"].astype(str) == "Yes").astype(int)
    tab = frame.assign(yes=yes).groupby(col, observed=False).yes.agg(["size", "mean"])
    tab.columns = ["n", "yes_rate"]
    return tab.reset_index().rename(columns={col: "level"}).assign(table=col)


cmp = pd.concat(
    [
        rates(train, "Subsidy_Available").assign(source="Playground"),
        rates(complete, "Subsidy_Available").assign(source="Original"),
        rates(train, "Range_Anxiety_Level").assign(source="Playground"),
        rates(complete, "Range_Anxiety_Level").assign(source="Original"),
    ],
    ignore_index=True,
)
fig = px.bar(
    cmp, x="level", y="yes_rate", color="source", facet_col="table", barmode="group",
    color_discrete_map={"Playground": TEAL, "Original": CORAL},
    title="Original rates are milder. Playground rates are a kill switch.",
)
fig.update_layout(**LAYOUT, yaxis_tickformat=".1%")
fig.update_yaxes(title="P(Yes)")
draw(fig)


Decision: extra original rows pull every nested target rate toward a softer machine. That is the opposite of more data.


<h2 id="3" style="color:#0B1F3A;border-bottom:3px solid #E85D4C;padding-bottom:6px;">3. Numerics disagree too</h2>

Question: is the mismatch only the two switches, or the whole generator?


In [ ]:
num_cols = [c for c in ("Annual_Income_USD", "Daily_Commute_km") if c in complete.columns]
pg_s = train[num_cols].assign(source="Playground").sample(n=min(40000, len(train)), random_state=42)
og_s = complete[num_cols].assign(source="Original")
both = pd.concat([pg_s, og_s], ignore_index=True)
fig = px.histogram(
    both, x="Annual_Income_USD", color="source", barmode="overlay", histnorm="probability density",
    color_discrete_map={"Playground": TEAL, "Original": CORAL},
    title="Income is not the same generator either (40k Playground sample).",
    opacity=0.65, nbins=60,
)
fig.update_layout(**LAYOUT)
draw(fig)


In [ ]:
if "Daily_Commute_km" in both.columns:
    fig = px.histogram(
        both, x="Daily_Commute_km", color="source", barmode="overlay",
        histnorm="probability density",
        color_discrete_map={"Playground": TEAL, "Original": CORAL},
        title="Commute kilometres: original 10k vs Playground.",
        opacity=0.65, nbins=50,
    )
    fig.update_layout(**LAYOUT)
    draw(fig)
miss = orig.isna().mean().rename("missing").reset_index().rename(columns={"index": "column"})
miss = miss.loc[miss["missing"] > 0].sort_values("missing", ascending=False)
if len(miss):
    fig = px.bar(miss, x="column", y="missing", title="Original missingness. Playground is zero.",
                 color_discrete_sequence=[CORAL])
    fig.update_layout(**LAYOUT, yaxis_tickformat=".0%")
    draw(fig)
else:
    print("this original file has no missing cells")


Decision: digits, income floors, and commute fractions are Playground artefacts. Copying original rows into those encodings is mixing two recipes.


<h2 id="4" style="color:#0B1F3A;border-bottom:3px solid #E85D4C;padding-bottom:6px;">4. Why extra labelled rows can hurt</h2>

Lab <code>exp025</code>: same frozen Stratified 5-fold, seed 42, same 3-seed LightGBM as the honest bag. Original rows concatenated onto fold-train only. One OOF prediction per Playground id. <b>0/5 folds.</b>


In [ ]:
lab = pd.DataFrame(
    {
        "file": ["honest bag", "+ original 10k", "public file stack"],
        "what": ["exp019 · 3-seed LightGBM", "exp025 · extra-fold original", "exp018 · nina high-band"],
        "cv": [0.946061, 0.946027, np.nan],
        "public": [0.94628, 0.94621, 0.94649],
        "folds_up": ["5/5 vs parent", "0/5 vs bag", "no OOF"],
        "use": ["honest hedge", "reject", "keep selected"],
    }
)
show(
    journal(
        lab, "The original table is allowed. It still lost.",
        "Cited from the frozen v1 log. This cell does not retrain exp025.",
        "Public 0.94649 is a rank mix of scored CSVs, not extra original rows.",
        rowname="file",
    )
    .fmt_number(["cv", "public"], decimals=6)
    .sub_missing(missing_text="—")
    .data_color("public", palette=HEAT)
)


In [ ]:
paired = pd.DataFrame(
    {
        "fold": [0, 1, 2, 3, 4],
        "bag 0.946061": [0.945110, 0.945802, 0.946968, 0.946359, 0.946085],
        "plus original": [0.945070, 0.945776, 0.946935, 0.946331, 0.946047],
    }
)
paired["delta"] = paired["plus original"] - paired["bag 0.946061"]
fig = go.Figure()
fig.add_bar(name="honest bag", x=paired.fold, y=paired["bag 0.946061"], marker_color=TEAL)
fig.add_bar(name="+ original 10k", x=paired.fold, y=paired["plus original"], marker_color=CORAL)
fig.update_layout(
    **LAYOUT, barmode="group",
    title="Every fold fell. Public 0.94621 vs bag 0.94628 vs file 0.94649.",
    yaxis=dict(title="ROC AUC", range=[0.9448, 0.9472]), xaxis_title="Frozen v1 fold",
)
draw(fig)


Decision: do not concatenate original rows hoping the switchboard averages out. Train Playground only. Do not mix that reject into a 0.94649 stack.


<h2 id="5" style="color:#0B1F3A;border-bottom:3px solid #E85D4C;padding-bottom:6px;">5. Train the Playground machine</h2>

Question: what is the strongest honest file this kernel can learn without the original 10k?

One LightGBM. Frozen Stratified 5-fold, seed 42. Encoders fit on fold-train only. Digits + smoothed target rates on income, commute, and the two switches. Original rows are not in <code>train</code>. CPU, expect ~15–25 minutes. Lab family: exp014/exp015, public <b>0.94616–0.94628</b>.


In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

SEED = 42
CATS = [
    "Gender", "City_Type", "Current_Car_Type",
    "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level",
]
NUMS = [
    "Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
LGBM = dict(
    n_estimators=3500, learning_rate=0.02, max_depth=5, num_leaves=32,
    min_child_samples=10, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.3, reg_alpha=0.071, reg_lambda=2.0, max_bin=255,
    random_state=SEED, n_jobs=4, verbose=-1, metric="auc",
)
assert orig is not train
print("training rows", len(train), "original rows held out", len(orig))


In [ ]:
def digits(frame: pd.DataFrame) -> pd.DataFrame:
    income = np.rint(frame["Annual_Income_USD"].to_numpy(float)).astype(np.int64)
    tenths = np.rint(frame["Daily_Commute_km"].to_numpy(float) * 10).astype(np.int64)
    commute = frame["Daily_Commute_km"].to_numpy(float)
    out = pd.DataFrame(index=frame.index)
    for p in (1, 10, 100, 1000):
        out[f"income_digit_{p}"] = (income // p) % 10
    for m in (10, 100, 1000):
        out[f"income_mod_{m}"] = income % m
        out[f"income_round_{m}"] = (income % m == 0).astype(np.int8)
    out["commute_tenth"] = tenths % 10
    out["commute_integer"] = np.floor(commute)
    out["commute_fraction"] = commute - np.floor(commute)
    return out


def te_keys(frame: pd.DataFrame) -> pd.DataFrame:
    income = frame["Annual_Income_USD"].astype(float)
    commute = frame["Daily_Commute_km"].astype(float)
    keys = pd.DataFrame({
        "income_exact": income.astype(str),
        "income_floor100": np.floor(income / 100).astype(str),
        "income_floor1000": np.floor(income / 1000).astype(str),
        "commute_exact": commute.astype(str),
        "commute_integer": np.floor(commute).astype(str),
        "switch": frame["Subsidy_Available"].astype(str) + "|" + frame["Range_Anxiety_Level"].astype(str),
    }, index=frame.index)
    for col in CATS:
        keys[col] = frame[col].astype(str)
    return keys


In [ ]:
def nested_rate(keys: pd.Series, y_col: pd.Series, m: float = 10.0) -> np.ndarray:
    out = np.zeros(len(keys), dtype=np.float32)
    splitter = KFold(5, shuffle=True, random_state=SEED)
    for fit_i, hold_i in splitter.split(keys):
        prior = float(y_col.iloc[fit_i].mean())
        stats = pd.DataFrame({"k": keys.iloc[fit_i], "y": y_col.iloc[fit_i]}).groupby("k")["y"].agg(["sum", "count"])
        mapped = (stats["sum"] + m * prior) / (stats["count"] + m)
        out[hold_i] = keys.iloc[hold_i].map(mapped).fillna(prior).to_numpy()
    return out


def oos_rate(tr_keys: pd.Series, y_col: pd.Series, out_keys: pd.Series, m=10.0):
    prior = float(y_col.mean())
    stats = pd.DataFrame({"k": tr_keys, "y": y_col}).groupby("k")["y"].agg(["sum", "count"])
    mapped = (stats["sum"] + m * prior) / (stats["count"] + m)
    return out_keys.map(mapped).fillna(prior).astype(np.float32)


def fold_xy(raw_tr, y_tr, raw_out):
    keys_tr, keys_out = te_keys(raw_tr), te_keys(raw_out)
    x = raw_out[NUMS + CATS].copy()
    for col in CATS:
        levels = sorted(raw_tr[col].astype(str).unique())
        x[col] = pd.Categorical(raw_out[col].astype(str), categories=levels)
    x = pd.concat([x, digits(raw_out)], axis=1)
    same = raw_out is raw_tr
    for col in keys_tr.columns:
        for m in (10.0, 100.0):
            tag = f"te_{col}_s{m:g}"
            x[tag] = nested_rate(keys_tr[col], y_tr, m) if same else oos_rate(keys_tr[col], y_tr, keys_out[col], m)
    return x


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(train), dtype=float)
test_sum = np.zeros(len(test), dtype=float)
fold_rows, imp_sum = [], None
for fold, (tr, va) in enumerate(folds.split(train, y)):
    raw_tr, raw_va = train.iloc[tr], train.iloc[va]
    y_tr, y_va = y.iloc[tr], y.iloc[va]
    X_tr, X_va, X_te = fold_xy(raw_tr, y_tr, raw_tr), fold_xy(raw_tr, y_tr, raw_va), fold_xy(raw_tr, y_tr, test)
    xtr, xes, ytr, yes = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)
    model = LGBMClassifier(**LGBM)
    model.fit(xtr, ytr, eval_set=[(xes, yes)], callbacks=[early_stopping(100), log_evaluation(200)])
    oof[va] = model.predict_proba(X_va)[:, 1]
    test_sum += model.predict_proba(X_te)[:, 1]
    auc = float(roc_auc_score(y_va, oof[va]))
    trees = int(getattr(model, "best_iteration_", 0) or 0)
    gain = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=X_tr.columns)
    imp_sum = gain if imp_sum is None else imp_sum.add(gain, fill_value=0)
    fold_rows.append({"fold": fold, "auc": auc, "trees": trees})
    print(f"fold {fold}  {auc:.6f}  trees {trees}", flush=True)
cv = float(roc_auc_score(y, oof))
trained = test_sum / 5
print("OOF", f"{cv:.6f}", "original rows used in fit: 0")


In [ ]:
fold_tbl = pd.DataFrame(fold_rows)
show(
    journal(
        fold_tbl,
        "Leak-free LightGBM on Playground only",
        f"OOF AUC {cv:.6f} · lab bag 0.946061 is the 3-seed ceiling, not this cell",
        "Original 10k never entered fold-train. Encoders never see fold-valid targets.",
        rowname="fold",
    )
    .cols_label(auc="Fold AUC", trees="Trees")
    .fmt_number("auc", decimals=6)
    .fmt_integer("trees")
    .data_color("auc", palette=HEAT)
)
top = imp_sum.sort_values(ascending=False).head(12).rename("gain").reset_index()
top.columns = ["feature", "gain"]
fig = px.bar(top, x="gain", y="feature", orientation="h",
             title="What the Playground model actually used", color_discrete_sequence=[TEAL])
fig.update_layout(**LAYOUT, yaxis={"categoryorder": "total ascending"})
draw(fig)


Decision: this is the honest file. Lab 3-seed bag public is <b>0.94628</b>. Mixing original 10k into this fit is the experiment that already lost.


<h2 id="6" style="color:#0B1F3A;border-bottom:3px solid #E85D4C;padding-bottom:6px;">6. Write the public file</h2>

Always write the trained ranking. For the official file, first look for Taeyang's exact notebook output: exp022 intended to use it, but a path bug left our 0.94649 file as primary. The source is accepted only from the named input and only if its IDs, zero-tie ordinal grid, and known output signature match. If absent, use the pinned 0.94649 high-band stack; otherwise use the model trained above.


In [ ]:
import hashlib


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_named(name: str):
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    hits = [p for p in root.rglob(name) if p.is_file()]
    return hits[0] if hits else None


BAND = [
    ("0.94649.csv", 0.58, "381e5c9d11e6e0cdc706a96cd0971f9b47bf4411f4d9923835927c477e5060ca"),
    ("0.94644.csv", 0.30, "277a6c3e680d82fb633b27901ce2e77d3d5a382587e094bb3864cafffa1658bc"),
    ("0.94639.csv", 0.12, "a3e40fcde1e555ef109c71d88ca55859802a563e1d277158b27f51c2181b4911"),
]


In [ ]:
def find_verified_top():
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    slug = "s6e9-094649-multi-paradigm-lexsort-master"
    hits = [p for p in root.rglob("submission.csv") if slug in str(p).lower()]
    if len(hits) > 1:
        raise RuntimeError(f"Multiple Taeyang outputs found: {hits}")
    return hits[0] if hits else None


top_path, top = find_verified_top(), None
if top_path is not None:
    raw_top = pd.read_csv(top_path)
    assert list(raw_top.columns) == ["id", "Will_Buy_EV"]
    assert raw_top["id"].is_unique
    aligned = sample[["id"]].merge(raw_top, on="id", how="left", validate="one_to_one")
    top = aligned["Will_Buy_EV"].to_numpy(float)
    assert np.isfinite(top).all() and len(np.unique(top)) == len(sample)
    expected_grid = (np.arange(len(sample)) + 0.5) / len(sample)
    assert np.allclose(np.sort(top), expected_grid, rtol=0, atol=2e-15)
    known = np.array([.568562, .477789, .306592, .247141, .555867,
                      .318303, .831660, .074619, .899479, .405238])
    assert np.allclose(top[:10], known, rtol=0, atol=6e-7)
    print("verified 0.94650 candidate", top_path)
    print("source sha256", file_sha256(top_path))
else:
    print("Taeyang output missing; using the best available fallback")


In [ ]:
inventory, found = [], []
for name, weight, pin in BAND:
    path = find_named(name)
    row = {"file": name, "weight": weight, "found": path is not None, "hash_ok": False}
    if path is not None and file_sha256(path) == pin:
        frame = pd.read_csv(path)
        assert list(frame.columns) == ["id", "Will_Buy_EV"]
        assert frame["id"].equals(sample["id"])
        row["hash_ok"] = True
        found.append((weight, frame))
    inventory.append(row)
show(
    journal(
        pd.DataFrame(inventory),
        "Community files as scored rankings — not extra train rows",
        "Pins are the lab hashes for nina 0.94649 / 0.94644 and Naji 0.94639.",
        "Original 10k is still held out. exp025 already rejected extra-train.",
        rowname="file",
    )
    .cols_label(weight="Blend weight", found="Found", hash_ok="Hash matches")
)


In [ ]:
def rank_map(frames, weights, reference):
    ranks = [w * f["Will_Buy_EV"].rank(method="average") for w, f in zip(weights, frames)]
    mixed = sum(ranks[1:], ranks[0])
    order = np.argsort(np.argsort(mixed.to_numpy()))
    return np.sort(reference["Will_Buy_EV"].to_numpy())[order]


def write_sub(path, values):
    out = Path("/kaggle/working") / path if Path("/kaggle/working").exists() else Path(path)
    sub = sample.copy()
    sub["Will_Buy_EV"] = values
    assert sub["id"].equals(sample["id"])
    assert np.isfinite(sub["Will_Buy_EV"]).all()
    sub.to_csv(out, index=False)
    return out, sub


trained_path, trained_sub = write_sub("submission_trained.csv", trained)
print("trained", trained_path, "mean", float(trained_sub["Will_Buy_EV"].mean()))
if top is not None:
    official, source = top, "Taeyang lexsort output (externally reported 0.94650)"
elif len(found) == 3:
    weights, frames = zip(*found)
    official, source = rank_map(frames, weights, frames[0]), "high-band rank stack mapped onto 0.94649"
elif len(found) == 1:
    official, source = found[0][1]["Will_Buy_EV"].to_numpy(), "single pinned high-band file"
else:
    official, source = trained, "trained Playground LightGBM (high-band files missing)"
out, sub = write_sub("submission.csv", official)
print("official", out, source, "mean", float(sub["Will_Buy_EV"].mean()))


Decision: two files. `submission_trained.csv` is the model this notebook learned on Playground. `submission.csv` is the public file. If it names Taeyang in the final log, expected public is 0.94650, but that score belongs to the cited ordering—not to this LightGBM. Do not blend original 10k into either one.


<div style="background:#0B1F3A;color:#F6F1E7;padding:22px 26px;border-radius:16px;margin-top:12px;">
<p style="letter-spacing:.14em;text-transform:uppercase;font-size:11px;opacity:.7;margin:0;">Reproduce</p>
<p style="margin:8px 0 0;">Competition: <a href="https://www.kaggle.com/competitions/playground-series-s6e9" style="color:#00B4A6;">playground-series-s6e9</a></p>
<p style="margin:6px 0 0;">Original table (diagnostic only): <a href="https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety" style="color:#00B4A6;">itzzomkar / EV adoption</a></p>
<p style="margin:6px 0 0;">Cited: <code>exp025</code> reject 0/5 · honest bag <b>0.94628</b> · high-band <b>0.94649</b> · Taeyang output reported <b>0.94650</b>.</p>
<p style="margin:6px 0 0;">Live: switchboard, numeric overlay, one LightGBM, verified external ordering with safe fallbacks.</p>
<p style="margin:6px 0 0;">Not used as train: original 10k, Zoom Zoom remix, lexsort, RealMLP.</p>
<p style="margin:14px 0 0;opacity:.75;font-size:13px;">Great Tables · Plotly · LightGBM. No vote-begging. No cloned blend.</p>
</div>
